## Preparación de Ambiente

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text storageName default "adlsftrsmardatadev01";

In [0]:
%python
storageName = dbutils.widgets.get("storageName")

## EXTERNAL LOCATIONS


In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extloc-ftr-smartdata-adlsmetastore-dev-01`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-ftr-smartdata-azure-dev-01`)
COMMENT 'Ubicación externa para las tablas de unit-catalog del data lake de retail';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extloc-ftr-smartdata-adlsraw-dev-01`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-ftr-smartdata-azure-dev-01`)
COMMENT 'Ubicación externa para las tablas raw del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extloc-ftr-smartdata-adlsbronze-dev-01`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-ftr-smartdata-azure-dev-01`)
COMMENT 'Ubicación externa para las tablas bronze del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extloc-ftr-smartdata-adlssilver-dev-01`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-ftr-smartdata-azure-dev-01`)
COMMENT 'Ubicación externa para las tablas silver del Data Lake';

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS `extloc-ftr-smartdata-adlsgold-dev-01`
URL 'abfss://gold@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `cred-ftr-smartdata-azure-dev-01`)
COMMENT 'Ubicación externa para las tablas gold del Data Lake';

## CATÁLOGO

In [0]:
%sql
DROP CATALOG IF EXISTS cat_ftr_smartdata_dev CASCADE;
     

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS cat_ftr_smartdata_dev
MANAGED LOCATION 'abfss://metastore@${storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';

## ESQUEMAS

In [0]:
%sql
DROP SCHEMA IF EXISTS cat_ftr_smartdata_dev.raw;
DROP SCHEMA IF EXISTS cat_ftr_smartdata_dev.bronze;
DROP SCHEMA IF EXISTS cat_ftr_smartdata_dev.silver;
DROP SCHEMA IF EXISTS cat_ftr_smartdata_dev.golden;

In [0]:
%python
dbutils.fs.rm(f"abfss://bronze@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://silver@{storageName}.dfs.core.windows.net/",True)
dbutils.fs.rm(f"abfss://gold@{storageName}.dfs.core.windows.net/",True)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS cat_ftr_smartdata_dev.raw;
CREATE SCHEMA IF NOT EXISTS cat_ftr_smartdata_dev.bronze;
CREATE SCHEMA IF NOT EXISTS cat_ftr_smartdata_dev.silver;
CREATE SCHEMA IF NOT EXISTS cat_ftr_smartdata_dev.golden;

## TABLAS BRONZE

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cat_ftr_smartdata_dev.bronze.ventas (
ID_ORDER STRING,
STORE STRING,
REG INT,
DATE DATE,
ID_ITEM STRING,
QTY INT,
PRICE DECIMAL(12,2),
DISCOUNT DECIMAL(12,2),
ID_SELLER STRING,
STATUS STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/ventas"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS cat_ftr_smartdata_dev.bronze.productos (
ID_ITEM STRING,
PRODUCT STRING,
CATEGORY STRING,
INGESTION_DATE TIMESTAMP
)
USING DELTA
LOCATION "abfss://bronze@${storageName}.dfs.core.windows.net/productos"



## TABLAS SILVER

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cat_ftr_smartdata_dev.silver.ventas_productos_categorias (
  ID_ARTICULO STRING,
  PRODUCTO STRING,
  CATEGORIA STRING,
  ID_PEDIDO STRING,
  TIENDA STRING,
  NUM_LINEA INT,
  FECHA DATE,
  CANTIDAD INT,
  PRECIO DOUBLE,
  DESCUENTO INT,
  ID_VENDEDOR STRING,
  ETIQUETA_DESCUENTO STRING
)
USING DELTA
LOCATION "abfss://silver@${storageName}.dfs.core.windows.net/ventas_productos_categorias"

##Tablas Gold

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cat_ftr_smartdata_dev.golden.ventas_diarias_tienda (
  TIENDA                    STRING,
  FECHA                     DATE,
  NUM_PEDIDOS               INT,
  TOTAL_UNIDADES            INT,
  INGRESO_BRUTO             DOUBLE,
  INGRESO_NETO              DOUBLE,
  DESCUENTO_MEDIO_PCT       DOUBLE,
  NUM_LINEAS_CON_DESCUENTO  INT,
  NUM_LINEAS_TOTAL          INT,
  PCT_LINEAS_CON_DESCUENTO  DOUBLE,
  FECHA_ACTUALIZACION       TIMESTAMP
)
USING DELTA
LOCATION "abfss://gold@${storageName}.dfs.core.windows.net/ventas_diarias_tienda"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS cat_ftr_smartdata_dev.golden.ventas_categoria_mes (
  CATEGORIA                 STRING,
  ANIO                      INT,
  MES                       INT,
  NUM_PEDIDOS               INT,
  NUM_ARTICULOS             INT,
  TOTAL_UNIDADES            INT,
  NUM_LINEAS_TOTAL          INT,
  INGRESO_BRUTO             DOUBLE,
  INGRESO_NETO              DOUBLE,
  TICKET_PROMEDIO           DOUBLE,
  PRECIO_PROMEDIO           DOUBLE,
  NUM_LINEAS_CON_DESCUENTO  INT,
  PCT_LINEAS_CON_DESCUENTO  DOUBLE,
  DESCUENTO_MEDIO_PCT       DOUBLE,
  NUM_PRECIO_NORMAL         INT,
  NUM_PROMOCION             INT,
  NUM_BONO_EMPRESARIAL      INT,
  NUM_REGALO                INT,
  FECHA_ACTUALIZACION       TIMESTAMP
)
USING DELTA
LOCATION "abfss://gold@${storageName}.dfs.core.windows.net/ventas_categoria_mes"
